In [11]:
# Instalar dependências (execute uma vez)
# !pip install google-genai pydantic

import os
import time
from google.colab import userdata
from google import genai
from pydantic import BaseModel, Field
from typing import Literal

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
MODEL_ID = "gemini-2.5-flash"

In [12]:
class Experiencia(BaseModel):
    empresa: str = Field(description="Nome da empresa.")
    cargo: str = Field(description="Cargo ou título do candidato nessa empresa.")
    periodo: str = Field(description="Período de trabalho, ex: '2019–2022'.")
    descricao: str = Field(description="Breve descrição das responsabilidades.")

class CurriculoEstruturado(BaseModel):
    nome: str = Field(description="Nome completo do candidato.")
    experiencias: list[Experiencia] = Field(
        description="Lista de experiências profissionais em ordem cronológica inversa."
    )
    habilidades_tecnicas: list[str] = Field(
        description="Habilidades técnicas mencionadas explicitamente no currículo."
    )
    senioridade: Literal["júnior", "pleno", "sênior", "staff/principal"] = Field(
        description=(
            "Nível de senioridade inferido a partir do tempo de experiência e "
            "complexidade dos cargos. Não extraído diretamente do texto."
        )
    )


In [13]:
CURRICULO = """
Lucas Mendes
lucas.mendes@email.com | linkedin.com/in/lucasmendes | São Paulo, SP

EXPERIÊNCIA PROFISSIONAL

Engenheiro de Machine Learning — NovaTech AI (2022–presente)
Liderança técnica de um time de 4 engenheiros no desenvolvimento de modelos de
recomendação para e-commerce. Implementação de pipelines de treinamento e serving
com PyTorch e Ray. Redução de latência de inferência em 40% via quantização e ONNX.

Cientista de Dados — Banco Meridional (2019–2022)
Desenvolvimento de modelos de crédito (XGBoost, LightGBM) para pessoa física e
jurídica. Criação de feature store em Spark. Trabalho próximo com times de risco
e regulatório para garantir explicabilidade dos modelos (SHAP, LIME).

Analista de Dados — StartLog (2017–2019)
Análise exploratória e dashboards em Python e Tableau. Automatização de relatórios
operacionais reduzindo trabalho manual em ~60%.

FORMAÇÃO
Bacharelado em Ciência da Computação — USP (2013–2017)

HABILIDADES TÉCNICAS
Python, PyTorch, TensorFlow, Scikit-learn, XGBoost, LightGBM, Spark, Ray,
SQL, dbt, Airflow, Docker, Kubernetes, ONNX, SHAP, Tableau
"""

In [14]:
def extrair_curriculo(curriculo: str) -> CurriculoEstruturado:
    prompt = f"""
Analise o currículo abaixo e extraia as informações solicitadas.

CURRÍCULO:
{curriculo}
"""
    for tentativa in range(3):
        try:
            resposta = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt,
                config={
                    "response_mime_type": "application/json",
                    "response_json_schema": CurriculoEstruturado.model_json_schema(),
                    "temperature": 0.1,
                }
            )
            return CurriculoEstruturado.model_validate_json(resposta.text)
        except Exception as e:
            if tentativa < 2:
                print(f"Tentativa {tentativa + 1} falhou, aguardando 10s...")
                time.sleep(10)
            else:
                raise e


In [15]:
VAGA = """
Engenheiro(a) de ML Sênior
- Experiência com sistemas de recomendação ou ranqueamento em produção
- Domínio de PyTorch ou TensorFlow
- Experiência com infraestrutura de ML (serving, monitoramento, pipelines)
- Diferenciais: Ray, ONNX, otimização de inferência
"""

resultado = extrair_curriculo(CURRICULO)

# Acessando os campos diretamente — sem parsing manual
print(f"Candidato: {resultado.nome}")
print(f"Senioridade: {resultado.senioridade}")
print(f"\nHabilidades ({len(resultado.habilidades_tecnicas)} encontradas):")
for h in resultado.habilidades_tecnicas:
    print(f"  - {h}")

print(f"\nExperiências:")
for exp in resultado.experiencias:
    print(f"  [{exp.periodo}] {exp.cargo} @ {exp.empresa}")


Candidato: Lucas Mendes
Senioridade: sênior

Habilidades (16 encontradas):
  - Python
  - PyTorch
  - TensorFlow
  - Scikit-learn
  - XGBoost
  - LightGBM
  - Spark
  - Ray
  - SQL
  - dbt
  - Airflow
  - Docker
  - Kubernetes
  - ONNX
  - SHAP
  - Tableau

Experiências:
  [2022–presente] Engenheiro de Machine Learning @ NovaTech AI
  [2019–2022] Cientista de Dados @ Banco Meridional
  [2017–2019] Analista de Dados @ StartLog


In [16]:
CURRICULO_2 = """
Fernanda Castro
fernanda.castro@email.com
Rio de Janeiro, RJ | linkedin.com/in/fernandacastro

== SOBRE MIM ==
Engenheira de ML com foco em sistemas de busca e ranking. Mestrado pela PUC-Rio.
Tenho experiência sólida com Python e gosto de trabalhar com PyTorch, além de
TorchServe para deploy. Já usei bastante Scikit-learn e XGBoost em projetos de
previsão de demanda. Conheço Feast para feature store, Kubeflow para pipelines,
MLflow para rastreamento de experimentos e Grafana para monitoramento. Trabalho
com Docker e Kubernetes no dia a dia, e SQL é essencial em quase tudo que faço.

* DataSphere (jan/2021 até hoje)
cargo: Engenheira de ML Sênior
Desenvolvo e mantenho modelos de ranking para busca em marketplace.
Treinamento com PyTorch e deploy via TorchServe. Responsável pela migração do
pipeline de treinamento para Kubeflow, reduzindo tempo de ciclo em 35%.

* Loggi — Engenheira de ML — 2018 a 2021
Modelos de previsão de demanda e otimização de rotas com Scikit-learn e XGBoost.
Construção de feature store em Feast. A/B testing de modelos em produção.

Boa Compra | Analista de Machine Learning | 2016-2018
Recomendação com filtragem colaborativa. Embeddings de produtos para busca semântica.

FORMAÇÃO:
PUC-Rio — Mestrado em Ciência da Computação, ênfase em ML (2014–2016)
PUC-Rio — Bacharelado em Engenharia de Computação (2010–2014)
"""


In [17]:
resultado_2 = extrair_curriculo(CURRICULO_2)

print(f"Candidato: {resultado_2.nome}")
print(f"Senioridade: {resultado_2.senioridade}")
print(f"\nHabilidades ({len(resultado_2.habilidades_tecnicas)} encontradas):")
for h in resultado_2.habilidades_tecnicas:
    print(f"  - {h}")

print(f"\nExperiências:")
for exp in resultado_2.experiencias:
    print(f"  [{exp.periodo}] {exp.cargo} @ {exp.empresa}")

Tentativa 1 falhou, aguardando 10s...
Candidato: Fernanda Castro
Senioridade: sênior

Habilidades (12 encontradas):
  - Python
  - PyTorch
  - TorchServe
  - Scikit-learn
  - XGBoost
  - Feast
  - Kubeflow
  - MLflow
  - Grafana
  - Docker
  - Kubernetes
  - SQL

Experiências:
  [jan/2021 até hoje] Engenheira de ML Sênior @ DataSphere
  [2018 a 2021] Engenheira de ML @ Loggi
  [2016-2018] Analista de Machine Learning @ Boa Compra


In [8]:
def triar_candidatos(
    curriculos: list[str],
) -> list[CurriculoEstruturado]:
    """Processa uma lista de currículos e retorna os candidatos estruturados."""
    resultados = []

    for i, curriculo in enumerate(curriculos):
        print(f"Processando candidato {i + 1}/{len(curriculos)}...")
        resultado = extrair_curriculo(curriculo)
        resultados.append(resultado)
        print(f"  ✓ {resultado.nome} — {resultado.senioridade}")

    return resultados


In [9]:
candidatos = triar_candidatos([CURRICULO, CURRICULO_2])

print(f"\n--- Resultado final: {len(candidatos)} candidato(s) processado(s) ---")
for c in candidatos:
    print(f"  {c.nome} | {c.senioridade} | {len(c.habilidades_tecnicas)} habilidades")


Processando candidato 1/2...
Tentativa 1 falhou, aguardando 10s...
  ✓ Lucas Mendes — sênior
Processando candidato 2/2...
  ✓ Fernanda Castro — sênior

--- Resultado final: 2 candidato(s) processado(s) ---
  Lucas Mendes | sênior | 16 habilidades
  Fernanda Castro | sênior | 12 habilidades
